# PatentFetcher — Lens.org Edition (OPV / MatScibert)

Fetches **complete patent text** (title + abstract + description + claims) for OPV donor/acceptor materials,
with emphasis on **processing-relevant fields** that feed MatScibert token extraction.

## Data Source
All patent data is retrieved from **[Lens.org](https://www.lens.org)** via its authenticated REST API.
Lens.org aggregates patents from USPTO, EPO, WIPO, and 90+ national offices.

## Pipeline
1. **Block 1** — Split `Active_Database.csv` → `donor_input.csv` / `acceptor_input.csv`
2. **Block 2** — Authenticate with Lens.org (Bearer token)
3. **Block 3** — Query Lens.org by SMILES / InChIKey to retrieve patent IDs + metadata
4. **Block 4** — Fetch full-text fields: title, abstract, description, claims, classifications
5. **Block 5** — Extract OPV processing-relevant sections for MatScibert
6. **Block 6** — Translate non-English fields (Google Translate fallback)
7. **Block 7** — Save corpus + diagnostics

## Fields Targeted for MatScibert
| Field | Why it matters for OPV processing |
|---|---|
| `title` | Material/device class signal |
| `abstract` | High-density processing keywords |
| `claims` | Precise process parameter ranges |
| `description` | Full fabrication protocols, solvent systems, annealing |
| `ipc_codes` | IPC H01L / C08G / C09K class filter |
| `processing_snippets` | Regex-extracted processing sentences |

> **Lens.org API key required.**  
> Get a free scholarly API token at https://www.lens.org/lens/user/subscriptions

In [9]:
# ──────────────────────────────────────────────────────────────
# BLOCK 0: Install / import dependencies
# ──────────────────────────────────────────────────────────────
!pip install -q requests pandas tqdm langdetect deep-translator rdkit-pypi

import os, re, time, json, hashlib
import pandas as pd
import requests
from tqdm.auto import tqdm
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator

# ── Google Drive mount (Colab) ──────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/OPV_Patents'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f'Drive mounted. Output folder: {DRIVE_ROOT}')
except ImportError:
    DRIVE_ROOT = '.'
    print('Not running in Colab — outputs saved locally.')

# ── Global paths ───────────────────────────────────────────────
INPUT_FILE   = 'Active_Database.csv'
DONOR_FILE   = 'donor_input.csv'
ACCEPTOR_FILE= 'acceptor_input.csv'
OUTPUT_FILE  = os.path.join(DRIVE_ROOT, 'opv_patent_corpus_lens.csv')
CACHE_DIR    = os.path.join(DRIVE_ROOT, 'lens_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Rate limiting ──────────────────────────────────────────────
LENS_DELAY   = 1.2   # seconds between Lens.org API calls
MAX_RETRIES  = 3

# Text fields to translate and feed to MatScibert
TEXT_FIELDS = ['title', 'abstract', 'claims', 'description']

print('Setup complete.')

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi
Not running in Colab — outputs saved locally.
Setup complete.


In [2]:
# ──────────────────────────────────────────────────────────────
# BLOCK 1: Split Active_Database.csv → donor / acceptor SMILES
# ──────────────────────────────────────────────────────────────
df_active = pd.read_csv(INPUT_FILE, encoding='latin1')

for col in ['Donor SMILES', 'Acceptor SMILES']:
    if col not in df_active.columns:
        raise ValueError(f"Column '{col}' not found in {INPUT_FILE}")

donor_df    = df_active[['Donor SMILES']].copy().rename(columns={'Donor SMILES': 'smiles'}).dropna()
acceptor_df = df_active[['Acceptor SMILES']].copy().rename(columns={'Acceptor SMILES': 'smiles'}).dropna()

donor_df.to_csv(DONOR_FILE, index=False)
acceptor_df.to_csv(ACCEPTOR_FILE, index=False)

print(f'Created {DONOR_FILE} with {len(donor_df)} entries.')
print(f'Created {ACCEPTOR_FILE} with {len(acceptor_df)} entries.')

Created donor_input.csv with 2439 entries.
Created acceptor_input.csv with 2439 entries.


In [10]:
# ──────────────────────────────────────────────────────────────
# BLOCK 2: Lens.org API authentication
#
# Set LENS_TOKEN as an environment variable or paste it below.
# Get a free token at: https://www.lens.org/lens/user/subscriptions
# (request 'Scholarly API' access — patent search is included)
# ──────────────────────────────────────────────────────────────

LENS_TOKEN = 'kR0zctOROBSuEMftK20uHGGijl5DNSrIeZh7hCMW7x7hsKzhywtb'

LENS_PATENT_URL = 'https://api.lens.org/patent/search'

LENS_HEADERS = {
    'Authorization': f'Bearer {LENS_TOKEN}',
    'Content-Type': 'application/json'
}

# ── Test authentication ────────────────────────────────────────
test_payload = {
    'query': {'match': {'title': 'organic photovoltaic'}},
    'size': 1,
    'include': ['lens_id', 'title']
}
r = requests.post(LENS_PATENT_URL, headers=LENS_HEADERS, json=test_payload)
if r.status_code == 200:
    print('✅ Lens.org authentication successful.')
    print(f"   Sample result: {r.json()['data'][0]['title']}")
elif r.status_code == 401:
    raise ValueError('❌ Invalid Lens.org token. Update LENS_TOKEN.')
else:
    print(f'⚠️  Unexpected status {r.status_code}: {r.text[:200]}')

⚠️  Unexpected status 400: {"reference":"6c07632d-0255-4663-9185-3bab9f3a6a7a","message":"Unrecognized fields - [title]","code":400}


In [13]:
# ──────────────────────────────────────────────────────────────
# BLOCK 3: Chemical resolution + Lens.org helper functions
#
# Resolution ladder (each step feeds the next as fallback):
#   SMILES  ──▶  PubChem CID  ──▶  InChIKey  ──▶  Lens.org query
#                     │                                    ▲
#                     └─── patent cross-ref list ──────────┘
#
# Why CID matters:
#   PubChem CID resolution validates that the SMILES is a real,
#   registered compound and provides a stable numeric ID.  The
#   CID→InChIKey path is more reliable than computing InChIKey
#   directly from raw SMILES because PubChem normalises tautomers,
#   stereochemistry, and salt forms first — giving a cleaner key
#   that matches what Lens.org stores in chemical_ids.inchikey.
#   PubChem also exposes a direct patent cross-reference list per
#   CID which we harvest as a Lens.org seed list.
# ──────────────────────────────────────────────────────────────
from urllib.parse import quote
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchiKey

PUBCHEM_BASE  = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
PUBCHEM_DELAY = 0.25   # PubChem rate limit: max 5 req/s

# Persistent HTTP session for PubChem (connection pooling)
pubchem_session = requests.Session()
pubchem_session.headers.update({'User-Agent': 'OPVPatentFetcher/2.0'})


# ── 1. SMILES → PubChem CID ───────────────────────────────────
def smiles_to_cid(smiles: str) -> int | None:
    """
    Resolve a SMILES string to its canonical PubChem CID.

    - URL-encodes the SMILES before sending (handles brackets,
      slashes, and stereo characters that break plain GET URLs)
    - Validates that the returned IdentifierList is non-empty
      before returning; returns None on any failure so the caller
      can cleanly fall back to the RDKit InChIKey path
    """
    smiles_encoded = quote(smiles, safe='')
    url = f'{PUBCHEM_BASE}/compound/smiles/{smiles_encoded}/cids/JSON'
    try:
        r = pubchem_session.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        id_list = data.get('IdentifierList', {})
        cids    = id_list.get('CID', [])
        if cids and len(cids) > 0:
            return int(cids[0])
    except requests.exceptions.HTTPError as e:
        code = e.response.status_code if e.response is not None else '?'
        if code not in (404, 400):   # log unexpected errors only
            print(f'  PubChem CID HTTP {code} for SMILES {smiles[:40]}...')
    except Exception as e:
        print(f'  PubChem CID error: {e}')
    finally:
        time.sleep(PUBCHEM_DELAY)
    return None


# ── 2. PubChem CID → InChIKey ─────────────────────────────────
def cid_to_inchikey(cid: int) -> str | None:
    """
    Fetch the canonical InChIKey for a PubChem CID.
    Uses PubChem's standardised structure, not the raw SMILES,
    so tautomers and salts are normalised before hashing.
    """
    url = f'{PUBCHEM_BASE}/compound/cid/{cid}/property/InChIKey/JSON'
    try:
        r = pubchem_session.get(url, timeout=30)
        r.raise_for_status()
        props = r.json().get('PropertyTable', {}).get('Properties', [])
        if props:
            return props[0].get('InChIKey')
    except Exception:
        pass
    finally:
        time.sleep(PUBCHEM_DELAY)
    return None


# ── 3. PubChem CID → patent ID list ──────────────────────────
def cid_to_patent_ids(cid: int) -> list[str]:
    """
    Fetch all patent IDs that PubChem associates with a CID via
    its cross-reference (xrefs) endpoint.

    Returns a list of patent strings like ['US8765432B2', 'WO2015123456A1']
    These are used as a direct seed list for Lens.org doc_key queries,
    which is more precise than a structure search when the compound
    is well-indexed in PubChem.
    """
    url = f'{PUBCHEM_BASE}/compound/cid/{cid}/xrefs/PatentID/JSON'
    try:
        r = pubchem_session.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        id_list = data.get('InformationList', {}).get('Information', [])
        if id_list:
            return id_list[0].get('PatentID', [])
    except requests.exceptions.HTTPError as e:
        code = e.response.status_code if e.response is not None else '?'
        if code not in (404, 400):
            print(f'  PubChem xref HTTP {code} for CID {cid}')
    except Exception:
        pass
    finally:
        time.sleep(PUBCHEM_DELAY)
    return []


# ── 4. RDKit fallback: SMILES → InChIKey ─────────────────────
def smiles_to_inchikey_rdkit(smiles: str) -> str | None:
    """
    Local RDKit fallback for SMILES → InChIKey when PubChem
    cannot resolve the structure (novel/unreported compounds).
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return MolToInchiKey(mol)
    except Exception:
        pass
    return None


# ── 5. Master resolution: SMILES → (cid, inchikey, patent_ids) ─
def resolve_smiles(smiles: str) -> dict:
    """
    Full resolution ladder for a SMILES string.
    Returns a dict with keys: cid, inchikey, pubchem_patent_ids.

    Resolution order:
      1. SMILES → PubChem CID  (preferred — normalises structure)
      2. CID    → InChIKey     (via PubChem standardised structure)
      3. CID    → patent IDs   (PubChem xrefs, seeds Lens.org lookup)
      4. Fallback: RDKit InChIKey if PubChem CID lookup fails
    """
    result = {'cid': None, 'inchikey': None, 'pubchem_patent_ids': []}

    cid = smiles_to_cid(smiles)
    if cid:
        result['cid']              = cid
        result['inchikey']         = cid_to_inchikey(cid)
        result['pubchem_patent_ids'] = cid_to_patent_ids(cid)
    else:
        # Novel/unregistered compound — compute InChIKey locally
        result['inchikey'] = smiles_to_inchikey_rdkit(smiles)

    return result


# ── Lens.org caching layer ────────────────────────────────────
def _cache_path(key: str) -> str:
    h = hashlib.md5(key.encode()).hexdigest()
    return os.path.join(CACHE_DIR, f'{h}.json')


def _cached_post(payload: dict) -> dict | None:
    cache_key = json.dumps(payload, sort_keys=True)
    cache_file = _cache_path(cache_key)

    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.post(
                LENS_PATENT_URL,
                headers=LENS_HEADERS,
                json=payload,
                timeout=30
            )

            if r.status_code == 200:
                data = r.json()
                with open(cache_file, "w") as f:
                    json.dump(data, f)
                return data

            elif r.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue

            else:
                print(f"Lens.org {r.status_code}: {r.text}")
                return None

        except Exception as e:
            print(f"Request error (attempt {attempt+1}): {type(e).__name__}: {e}")
            time.sleep(3)

    return None
# ── Fields to retrieve from Lens.org ──────────────────────────
LENS_INCLUDE_FIELDS = [
    "lens_id",
    "doc_key",
    "jurisdiction",
    "doc_number",
    "kind",
    "date_published",
    "publication_type",
    "biblio",
    "abstract",
    "claims",
    "description",
    "legal_status",
]


# ── Lens.org query functions ──────────────────────────────────
def _pubchem_id_to_doc_key(patent_id: str) -> str:
    """
    Convert a PubChem patent ID to Lens.org doc_key format.
    PubChem returns IDs like 'US-11690282-B2'; Lens doc_key uses 'US_11690282_B2'.
    """
    return patent_id.strip().replace('-', '_')


def search_lens_by_patent_ids(patent_ids: list[str], size: int = 50) -> list[dict]:
    results = []
    for i in range(0, len(patent_ids), 50):
        batch = [_pubchem_id_to_doc_key(p) for p in patent_ids[i:i + 50]]
        payload = {
            "query": {
                "terms": {
                    "ids": batch
                }
            },
            "size": min(size, 50),
            "include": LENS_INCLUDE_FIELDS
        }
        data = _cached_post(payload)
        if data and "data" in data:
            results.extend(data["data"])
    return results


def search_lens_for_smiles(smiles: str, resolved: dict,
                           size: int = 10) -> list[dict]:
    """
    Lens.org search using PubChem patent xrefs → Lens doc_key lookup.
    Returns empty list if no patent xrefs are available.
    No request is made if pubchem_patent_ids is empty.
    """
    if not resolved['pubchem_patent_ids']:
        return []

    seen, results = set(), []

    def _add(records):
        for rec in records:
            lid = rec.get('lens_id', '')
            if lid and lid not in seen:
                seen.add(lid)
                results.append(rec)

    _add(search_lens_by_patent_ids(resolved['pubchem_patent_ids'], size=size))
    return results


def extract_text(field_value) -> str:
    """
    Lens.org returns title/abstract/description/claims as either a string
    or a list of dicts with 'text' keys. Normalize to plain string.
    """
    if not field_value:
        return ""

    if isinstance(field_value, str):
        return field_value.strip()

    if isinstance(field_value, dict):
        # abstract / description
        if "text" in field_value and isinstance(field_value["text"], str):
            return field_value["text"].strip()

        # title/name-like objects
        if "value" in field_value and isinstance(field_value["value"], str):
            return field_value["value"].strip()

        # claims object
        if "claims" in field_value:
            return extract_text(field_value["claims"])

        # claim_text list
        if "claim_text" in field_value:
            return extract_text(field_value["claim_text"])

        return ""
    if isinstance(field_value, list):
        parts = []
        for item in field_value:
            txt = extract_text(item)
            if txt:
                parts.append(txt)
        return "\n".join(parts).strip()

    return str(field_value).strip()


def extract_ipc_codes(classifications) -> str:
    """
    Extract IPC classification codes, prioritizing OPV-relevant classes:
      H01L 51  — organic semiconductor devices
      C08G 61  — conjugated polymers
      C09K 11  — luminescent materials
    """
    if not classifications:
        return ''
    codes = []
    for cl in (classifications if isinstance(classifications, list) else [classifications]):
        if isinstance(cl, dict):
            code = cl.get('code', '') or cl.get('symbol', '')
            if code:
                codes.append(code.strip())
        elif isinstance(cl, str):
            codes.append(cl.strip())
    return '; '.join(codes)


def parse_lens_record(record: dict, smiles: str, role: str,
                         resolved: dict | None = None) -> dict:
    """
    Flatten a Lens.org patent record into a row dict with
    all processing-relevant text fields for MatScibert.
    Stores pubchem_cid and inchikey from the resolved dict so the
    corpus is traceable back to the source chemical identity.
    """
    title       = extract_text(record.get('title'))
    abstract    = extract_text(record.get('abstract'))
    description = extract_text(record.get('description'))
    claims      = extract_text(record.get('claims'))
    ipc_codes   = extract_ipc_codes(record.get('classifications'))

    # Biblio fallback for title
    if not title:
        biblio = record.get('biblio', {})
        title = extract_text(biblio.get('invention_title', []))

    full_text = '\n\n'.join(t for t in [title, abstract, claims, description] if t)

    jurisdiction = record.get('jurisdiction', '')
    if not jurisdiction:
        doc_key = record.get('doc_key', '')
        jurisdiction = doc_key[:2] if doc_key else ''

    resolved = resolved or {}
    return {
        'smiles':              smiles,
        'role':                role,
        'pubchem_cid':         resolved.get('cid', ''),
        'inchikey':            resolved.get('inchikey', ''),
        'lens_id':             record.get('lens_id', ''),
        'doc_key':             record.get('doc_key', ''),
        'jurisdiction':        jurisdiction,
        'year':                record.get('year_published', ''),
        'pub_type':            record.get('publication_type', ''),
        'ipc_codes':           ipc_codes,
        'title':               title,
        'abstract':            abstract,
        'claims':              claims,
        'description':         description,
        'full_text':           full_text,
        'translated':          False,
        'source':              'Lens.org'
    }


print('Chemical resolution + Lens.org helper functions defined.')

Chemical resolution + Lens.org helper functions defined.


In [ ]:
# ──────────────────────────────────────────────────────────────
# BLOCK 4a: Resolve SMILES → PubChem CID / InChIKey / Patent IDs
#
# For every donor and acceptor SMILES, run the resolution ladder:
#   1. SMILES → PubChem CID
#   2. CID    → InChIKey
#   3. CID    → PubChem patent xrefs
#   4. RDKit fallback for InChIKey if PubChem fails
#
# Saves one row per unique SMILES to RESOLUTION_FILE and prints
# coverage statistics so you know what to expect before hitting
# the Lens.org API.
# ──────────────────────────────────────────────────────────────

RESOLUTION_FILE = 'smiles_resolution.csv'

resolution_rows = []
no_cid      = 0
no_resolve  = 0
no_patents  = 0

for role, file in [('donor', DONOR_FILE), ('acceptor', ACCEPTOR_FILE)]:
    df_input = pd.read_csv(file)

    if 'smiles' not in df_input.columns:
        raise ValueError(f"{file} does not contain a 'smiles' column")

    smiles_list = (
        df_input['smiles']
        .dropna()
        .astype(str)
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .unique()
        .tolist()
    )

    print(f'\n=== {role.upper()} — {len(smiles_list)} unique SMILES ===')

    for smi in tqdm(smiles_list, desc=f'{role} resolution'):
        resolved = resolve_smiles(smi)

        has_cid      = bool(resolved.get('cid'))
        has_inchikey = bool(resolved.get('inchikey'))
        has_patents  = bool(resolved.get('pubchem_patent_ids'))

        if not has_cid:
            no_cid += 1
        if not has_inchikey:
            no_resolve += 1
            tqdm.write(f'  No resolution for SMILES: {smi[:80]}')
        if not has_patents:
            no_patents += 1

        resolution_rows.append({
            'smiles'            : smi,
            'role'              : role,
            'pubchem_cid'       : resolved.get('cid', ''),
            'inchikey'          : resolved.get('inchikey', ''),
            'pubchem_patent_ids': ';'.join(resolved.get('pubchem_patent_ids', [])),
        })

df_resolution = pd.DataFrame(resolution_rows)
df_resolution.to_csv(RESOLUTION_FILE, index=False)

total        = len(df_resolution)
with_cid     = df_resolution['pubchem_cid'].astype(str).str.strip().replace('', pd.NA).notna().sum()
with_patents = (df_resolution['pubchem_patent_ids'].astype(str).str.strip() != '').sum()
unresolvable = (df_resolution['inchikey'].astype(str).str.strip() == '').sum()

print(f'\n=== Resolution complete — saved to {RESOLUTION_FILE} ===')
print(f'Total unique SMILES          : {total}')
print(f'  With PubChem CID           : {with_cid}  ({100*with_cid/total:.1f}%)')
print(f'  With PubChem patent xrefs  : {with_patents}  ({100*with_patents/total:.1f}%)')
print(f'  Unresolvable (no InChIKey) : {unresolvable}  ({100*unresolvable/total:.1f}%)')


TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [16]:
# ──────────────────────────────────────────────────────────────
# BLOCK 4b: Fetch patents from Lens.org using resolved SMILES
#
# Reads RESOLUTION_FILE produced by BLOCK 4a.
# For each SMILES that has at least an InChIKey, queries Lens.org:
#   1. PubChem patent IDs → Lens doc_key lookup   (most precise)
#   2. InChIKey           → Lens chemical_ids      (structure search)
# ──────────────────────────────────────────────────────────────

PATENTS_PER_SMILES = 10
SAVE_INTERVAL      = 50

df_resolution = pd.read_csv(RESOLUTION_FILE)
# Keep only rows that are resolvable (have an InChIKey)
df_resolvable = df_resolution[df_resolution['pubchem_patent_ids'].notna()].reset_index(drop=True)

print(f'Loaded {len(df_resolution)} SMILES from {RESOLUTION_FILE}')
print(f'  With PubChem patent xrefs  : {len(df_resolvable)}')
print(f'  Skipped (no patent xrefs)  : {len(df_resolution) - len(df_resolvable)}')

all_rows  = []
seen_docs = set()
no_patents_found = 0

for i, row in enumerate(tqdm(df_resolvable.itertuples(index=False), total=len(df_resolvable), desc='Lens fetch'), start=1):
    smi = row.smiles

    resolved = {
        'cid'               : row.pubchem_cid if pd.notna(row.pubchem_cid) and str(row.pubchem_cid).strip() else None,
        'inchikey'          : str(row.inchikey).strip() or None,
        'pubchem_patent_ids': [p for p in str(row.pubchem_patent_ids).split(';') if p.strip()],
    }

    tqdm.write(f'  SMILES: {smi[:50]}')
    tqdm.write(f'  Patent IDs ({len(resolved["pubchem_patent_ids"])}): {resolved["pubchem_patent_ids"]}')
    records = search_lens_for_smiles(smi, resolved, size=PATENTS_PER_SMILES)

    if not records:
        no_patents_found += 1
    else:
        added = 0
        for rec in records:
            unique_key = rec.get('lens_id') or rec.get('doc_key')
            if unique_key and unique_key in seen_docs:
                continue
            all_rows.append(parse_lens_record(rec, smi, row.role, resolved))
            added += 1
            if unique_key:
                seen_docs.add(unique_key)

        if added == 0:
            tqdm.write(f'  Duplicate-only hits for {smi[:50]}...')

    time.sleep(LENS_DELAY)

    if i % SAVE_INTERVAL == 0:
        pd.DataFrame(all_rows).to_csv(OUTPUT_FILE, index=False)
        print(f'  Checkpoint: {len(all_rows)} rows at SMILES {i}/{len(df_resolvable)}')

df_corpus = pd.DataFrame(all_rows)
df_corpus.to_csv(OUTPUT_FILE, index=False)

unique_lens_ids = df_corpus['lens_id'].nunique() if 'lens_id' in df_corpus.columns and not df_corpus.empty else 0

print(f'\n=== Lens fetch complete ===')
print(f'Total patent rows  : {len(df_corpus)}')
print(f'Unique Lens IDs    : {unique_lens_ids}')
print(f'SMILES → no patents: {no_patents_found}')
print(f'Saved to {OUTPUT_FILE}')


Loaded 1816 SMILES from smiles_resolution.csv
  With PubChem patent xrefs  : 49
  Skipped (no patent xrefs)  : 1767


Lens fetch:   0%|          | 0/49 [00:00<?, ?it/s]

Lens fetch:   0%|          | 0/49 [00:00<?, ?it/s]

  SMILES: CCCCC(CC)COC(=O)c1sc2c(-c3cc4c(-c5ccc(CC(CC)CCCC)s
  Patent IDs (11): ['US-11690282-B2', 'US-11773210-B2', 'US-20140378605-A1', 'US-2014378605-A1', 'US-20150210800-A1', 'US-20170033304-A1', 'US-2019245150-A1', 'US-2019378988-A1', 'US-20230295494-A1', 'US20140378605', 'US20150210800']


Lens fetch:   2%|▏         | 1/49 [00:01<00:58,  1.21s/it]

...

  SMILES: CCCCC(CC)CN1C(=O)c2ccc(C#Cc3ccc(C#Cc4ccc5c(c4)C(=O
  Patent IDs (1): ['JP-2014047192-A']


Lens fetch:  98%|█████████▊| 48/49 [31:31<00:41, 41.62s/it]

  SMILES: CCCCC(CC)CN1C(=O)C2=C(C3=CC=C(C4=CC=C(C5=CC=C(C6=C
  Patent IDs (16): ['CN-113224115-B', 'CN-113281802-B', 'CN-115144888-B', 'EP-3863059-A1', 'EP-3863059-B1', 'EP-4068363-A1', 'EP-4068363-B1', 'EP-4180845-A1', 'US-11573336-B2', 'US-11581359-B2', 'US-12433044-B2', 'US-20210239859-A1', 'US-20210242273-A1', 'US-2021242273-A1', 'US-20220359128-A1', 'US-20230145517-A1']


Lens fetch: 100%|██████████| 49/49 [31:38<00:00, 38.75s/it]



=== Lens fetch complete ===
Total patent rows  : 2714
Unique Lens IDs    : 2714
SMILES → no patents: 0
Saved to ./opv_patent_corpus_lens.csv


In [ ]:
# ──────────────────────────────────────────────────────────────
# BLOCK 4b: Supplemental OPV keyword search on Lens.org
#
# Broadens the corpus with patents describing OPV fabrication
# processes that may not match specific SMILES in the database.
# These are high-value for MatScibert processing constraint tokens.
# ──────────────────────────────────────────────────────────────

OPV_QUERIES = [
    # Device fabrication
    'organic photovoltaic bulk heterojunction fabrication',
    'perovskite organic solar cell processing',
    'non-fullerene acceptor organic solar cell',
    # Solvent/annealing processing
    'solvent annealing organic semiconductor thin film',
    'thermal annealing bulk heterojunction morphology',
    'chlorobenzene dichlorobenzene donor acceptor blend',
    'spin coating organic photovoltaic active layer',
    # Interfacial layers
    'PEDOT:PSS hole transport layer organic solar',
    'zinc oxide electron transport layer inverted OPV',
    'MoO3 anode interlayer organic photovoltaic',
    # Material classes
    'P3HT PCBM blend morphology annealing',
    'Y6 BTP-eC9 non-fullerene acceptor processing',
    'PTQ10 PM6 ternary blend organic solar cell',
    'conjugated polymer donor bulk heterojunction synthesis',
    'small molecule acceptor fused ring processing',
]

supplemental_rows = []
supp_seen = set(seen_ids)  # avoid duplicating SMILES-matched patents

for query in tqdm(OPV_QUERIES, desc='OPV keyword queries'):
    test_payload = {
    "query": {
        "match": {
            "abstract": "organic photovoltaic"
        }
    },
    "size": 3,
    "include": [
        "lens_id",
        "biblio.invention_title",
        "abstract",
        "biblio.classifications_ipcr"
    ]
}
    payload = {
    "query": {
        "bool": {
            "must": [
                {
                    "query_string": {
                        "query": query,
                        "fields": ["title", "abstract", "claims", "description"],
                        "default_operator": "OR"
                    }
                }
            ],
            "filter": [
                {
                    "terms": {
                        "biblio.classifications_ipcr.classifications.symbol": [
                            "H01L51",
                            "H01L31",
                            "C08G61",
                            "C08L65",
                            "C09K11"
                        ]
                    }
                }
            ]
        }
    },
    "size": 20,
    "include": LENS_INCLUDE_FIELDS
}

    result = _cached_post(payload=payload)

    if result and "data" in result:
        for rec in result["data"]:
            lid = rec.get("lens_id", "")
            if lid in supp_seen:
                continue
            row = parse_lens_record(rec, smiles="KEYWORD_SEARCH", role="supplemental")
            row["keyword_query"] = query
            supplemental_rows.append(row)
            if lid:
                supp_seen.add(lid)

    time.sleep(LENS_DELAY)
    
print(f'Supplemental OPV patents found: {len(supplemental_rows)}')

# Merge into corpus
if supplemental_rows:
    df_supp = pd.DataFrame(supplemental_rows)
    df_corpus = pd.concat([df_corpus, df_supp], ignore_index=True)
    df_corpus.to_csv(OUTPUT_FILE, index=False)
    print(f'Total corpus size after supplement: {len(df_corpus)}')

In [ ]:
# ──────────────────────────────────────────────────────────────
# BLOCK 4c: High-recall supplemental OPV keyword search on Lens.org
#
# Broadens the corpus with OPV/device/process patents that may not
# be captured by exact SMILES/InChIKey matching.
# ──────────────────────────────────────────────────────────────
OPV_QUERY_SPECS = [
    {"name": "core_opv", "text": "organic solar cell"},
    {"name": "organic_photovoltaic", "text": "organic photovoltaic"},
    {"name": "bulk_heterojunction", "text": "bulk heterojunction"},
    {"name": "non_fullerene_acceptor", "text": "non-fullerene acceptor"},
    {"name": "polymer_solar_cell", "text": "polymer solar cell"},
]

RESULTS_PER_QUERY = 10

supplemental_rows = []
supp_seen = set()   # debug: do not dedupe against old results yet

for spec in tqdm(OPV_QUERY_SPECS, desc="OPV keyword queries"):
    payload = {
        "query": {
            "match": {
                "abstract": spec["text"]
            }
        },
        "size": RESULTS_PER_QUERY,
        "include": ["lens_id", "doc_key", "biblio", "abstract", "claims", "description"],
        "sort": [{"date_published": "desc"}]
    }

    result = _cached_post(payload)

    if not result:
        print(f"\n[{spec['name']}] no response")
        continue

    records = result.get("data", [])
    print(f"\n[{spec['name']}] returned={len(records)}")

    for rec in records:
        unique_key = rec.get("lens_id") or rec.get("doc_key")
        if unique_key and unique_key in supp_seen:
            continue

        row = parse_lens_record(rec, smiles="KEYWORD_SEARCH", role="supplemental")
        row["keyword_query"] = spec["name"]
        supplemental_rows.append(row)

        if unique_key:
            supp_seen.add(unique_key)

    time.sleep(LENS_DELAY)

print(f"Supplemental OPV patents found: {len(supplemental_rows)}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# BLOCK 5: Extract OPV processing-relevant snippets
#
# This block parses full_text / description / claims and pulls
# sentences containing processing constraint keywords.
# These snippets are the primary input for MatScibert NER/RE.
#
# Processing constraint categories targeted:
#   - Solvent systems (chlorobenzene, o-DCB, THF, toluene, etc.)
#   - Concentration / solution parameters (mg/mL, wt%, v/v)
#   - Deposition method (spin-coat, blade-coat, slot-die, vapor deposit)
#   - Annealing (temperature, duration, atmosphere)
#   - Film thickness (nm)
#   - Additive (DIO, CN, ITIC, PDI)
#   - Electrode/interlayer deposition
#   - D:A ratio, blend composition
# ──────────────────────────────────────────────────────────────
import re

# ── Processing keyword regex patterns ──────────────────────────
PROCESSING_PATTERNS = [
    # Solvents
    r'\b(chlorobenzene|dichlorobenzene|o-DCB|chloroform|toluene|'
     r'xylene|tetrahydrofuran|THF|dimethylformamide|DMF|'
     r'dimethyl sulfoxide|DMSO|N-methyl-2-pyrrolidone|NMP|'
     r'anisole|mesitylene|1,2,4-trimethylbenzene)\b',
    # Deposition methods
    r'\b(spin.coat|blade.coat|slot.die|doctor.blade|'
     r'inkjet|gravure|screen.print|thermal evapor|'
     r'vacuum deposit|vapor deposit|roll.to.roll)\b',
    # Annealing
    r'\b(anneal|anneal\w*|thermal treatment|heat treatment)\b'
     r'.*?\d+\s*°?\s*[Cc°]',
    # Temperature standalone
    r'\d{2,3}\s*°?\s*[Cc°]\b',
    # Concentration
    r'\d+\.?\d*\s*(mg/mL|mg mL|wt%|vol%|v/v|w/w|M\b)',
    # Thickness
    r'\d+\.?\d*\s*nm\b',
    # D:A ratio
    r'\b(donor.to.acceptor|D:A|weight ratio|blend ratio)\b'
     r'.*?\d',
    # Additive
    r'\b(additive|DIO|1,8-diiodooctane|CN|chloronaphthalene|'
     r'pyridine additive|solvent additive)\b',
    # Spin speed/time
    r'\d+\s*(rpm|r\.p\.m|rcf)\b',
    r'\d+\s*s(ec|econds)?\s*at\s*\d+',
    # Atmosphere
    r'\b(nitrogen|argon|glove.?box|inert atmosphere|'
     r'air.free|vacuum|ambient)\b',
    # General processing section headers
    r'\b(fabricat|process|preparation|deposited|cast from|'
     r'dissolved in|filtered through|stirred at|heated to)\w*\b',
]

COMBINED_PATTERN = re.compile(
    '|'.join(f'(?:{p})' for p in PROCESSING_PATTERNS),
    re.IGNORECASE
)

SENTENCE_SPLIT = re.compile(r'(?<=[.!?])\s+')


def extract_processing_snippets(text: str, window: int = 2) -> str:
    """
    Split text into sentences, return sentences (plus ±window context)
    that match processing keywords.
    """
    if not text or len(text) < 20:
        return ''
    sentences = SENTENCE_SPLIT.split(text)
    hits = set()
    for i, sent in enumerate(sentences):
        if COMBINED_PATTERN.search(sent):
            for j in range(max(0, i - window), min(len(sentences), i + window + 1)):
                hits.add(j)
    if not hits:
        return ''
    return ' '.join(sentences[j] for j in sorted(hits))


# Apply extraction across corpus
df_corpus = pd.read_csv(OUTPUT_FILE)

tqdm.pandas(desc='Extracting processing snippets')

df_corpus['processing_snippets'] = df_corpus.apply(
    lambda row: extract_processing_snippets(
        ' '.join(str(row.get(f, '')) for f in ['description', 'claims', 'abstract'])
    ),
    axis=1
)

has_snippets = df_corpus['processing_snippets'].str.len().gt(20)
print(f'Patents with processing snippets: {has_snippets.sum()} / {len(df_corpus)}')
print(f'\nSample snippet:\n{df_corpus.loc[has_snippets, "processing_snippets"].iloc[0][:500]}')

df_corpus.to_csv(OUTPUT_FILE, index=False)

In [ ]:
# ──────────────────────────────────────────────────────────────
# BLOCK 6: Robust sentence-aware translation of non-English fields
#
# Why this matters for MatScibert:
#   MatScibert was trained on English materials-science text.
#   Any non-English sentence in description/claims/snippets will
#   tokenize incorrectly and produce garbage NER predictions.
#
# Design decisions:
#   1. SENTENCE-BOUNDARY CHUNKING — splits on sentence endings so
#      Google Translate avoids truncated sentences. If a single
#      run-on sentence exceeds the 4800-char limit, the code safely
#      falls back to word-boundary splitting — never mid-character.
#      Character-level slicing (old approach) broke mid-formula
#      and mid-clause, corrupting translated output.
#
#   2. ROBUST LANGUAGE DETECTION — langdetect alone is unreliable
#      on patent text full of chemical names, numbers, and mixed
#      scripts. We use a two-stage check:
#        a) langdetect majority vote over 3 probes (seeded)
#        b) Unicode script scan to catch CJK/Cyrillic/Arabic even
#           when langdetect falsely returns 'en'
#
#   3. EXPONENTIAL BACKOFF — Google Translate's free tier rate-
#      limits aggressively. Retries with 2^n backoff + jitter.
#
#   4. AUDIT COLUMNS — adds `detected_lang` and `translated_fields`
#      so you can verify what was changed and filter by language.
#
#   5. ORDER: translation runs BEFORE processing_snippets re-
#      extraction at the end of this block, so snippets are always
#      drawn from English text.
# ──────────────────────────────────────────────────────────────
import random
import unicodedata
from langdetect import detect, detect_langs, LangDetectException
from langdetect import DetectorFactory
from deep_translator import GoogleTranslator

DetectorFactory.seed = 42  # make langdetect deterministic

# Fields to translate, in order of length (short fields first = faster
# failures if quota is exhausted, preserving the most-used fields)
TRANSLATE_FIELDS = ['title', 'abstract', 'claims', 'description']

# Google Translate free tier: ~5000 chars per request. We stay under
# 4800, splitting on sentence boundaries where possible; run-on sentences
# that exceed the limit fall back to word-boundary splits, never mid-char.
CHUNK_CHAR_LIMIT = 4800

# Non-Latin Unicode blocks that guarantee non-English content even
# when langdetect mis-identifies the language as English.
NON_LATIN_RANGES = [
    (0x0400, 0x04FF),   # Cyrillic (Russian, Ukrainian…)
    (0x0600, 0x06FF),   # Arabic
    (0x3000, 0x9FFF),   # CJK (Chinese, Japanese, Korean)
    (0xAC00, 0xD7AF),   # Hangul
    (0x0370, 0x03FF),   # Greek
]


def _contains_non_latin(text: str, threshold: float = 0.05) -> bool:
    """
    Return True if >threshold fraction of alphabetic characters fall
    in non-Latin Unicode blocks.  Catches CJK/Cyrillic that langdetect
    sometimes misses in short or mixed patent strings.
    """
    alpha_chars = [c for c in text if c.isalpha()]
    if not alpha_chars:
        return False
    non_latin = sum(
        1 for c in alpha_chars
        if any(lo <= ord(c) <= hi for lo, hi in NON_LATIN_RANGES)
    )
    return (non_latin / len(alpha_chars)) > threshold


def detect_language(text: str) -> str:
    """
    Robustly detect the dominant language of a text string.
    Returns ISO 639-1 code (e.g. 'en', 'de', 'zh-cn', 'ja').
    Falls back to 'en' only if detection genuinely fails.

    Uses a 500-char sample from the MIDDLE of the text to avoid
    patent boilerplate headers ('This invention relates to…')
    which are often English even in non-English patents.
    """
    text = text.strip()
    if len(text) < 20:
        return 'en'

    # Sample from middle 500 chars to skip English boilerplate
    mid = len(text) // 2
    sample = text[max(0, mid - 250): mid + 250]

    # Fast path: script scan catches CJK/Cyrillic/Arabic unambiguously
    if _contains_non_latin(sample):
        # Confirm with langdetect for the exact code
        try:
            return detect(sample)
        except LangDetectException:
            return 'und'  # undetermined but non-English

    # For Latin-script languages (German, French, etc.) use majority
    # vote across three probe positions to reduce false 'en' returns
    votes = []
    for offset in [0, len(text) // 3, 2 * len(text) // 3]:
        probe = text[offset: offset + 400]
        if len(probe.split()) < 5:
            continue
        try:
            votes.append(detect(probe))
        except LangDetectException:
            pass

    if not votes:
        return 'en'
    # Return the most common vote
    return max(set(votes), key=votes.count)


def _split_into_sentence_chunks(text: str, char_limit: int = CHUNK_CHAR_LIMIT) -> list[str]:
    """
    Split text into chunks that never exceed char_limit characters,
    breaking on sentence boundaries (. ! ? followed by whitespace)
    wherever possible.

    If a single run-on sentence already exceeds char_limit on its own,
    it falls back to word-boundary splitting so Google Translate still
    receives valid UTF-8 tokens rather than a mid-character cut.
    This is preferable to the old character-slice approach, which broke
    mid-formula and mid-clause and corrupted translated procedure text.
    """
    # Split into sentences, keeping delimiters
    raw_sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ''
    for sent in raw_sentences:
        # If a single sentence is already over the limit, hard-split
        # only at whitespace (never mid-word or mid-formula)
        if len(sent) > char_limit:
            if current:
                chunks.append(current.strip())
                current = ''
            words = sent.split(' ')
            buf = ''
            for w in words:
                if len(buf) + len(w) + 1 > char_limit:
                    if buf:
                        chunks.append(buf.strip())
                    buf = w
                else:
                    buf = (buf + ' ' + w).strip()
            if buf:
                current = buf
        elif len(current) + len(sent) + 1 > char_limit:
            chunks.append(current.strip())
            current = sent
        else:
            current = (current + ' ' + sent).strip()
    if current:
        chunks.append(current.strip())
    return [c for c in chunks if c]


def translate_to_english(text: str, source_lang: str = 'auto',
                         max_retries: int = 4) -> str:
    """
    Translate text to English using Google Translate via deep-translator.

    - Chunks on sentence boundaries; falls back to word boundaries
      for run-on sentences that exceed the character limit
    - Exponential backoff with jitter on rate-limit errors
    - Falls back to original chunk on unrecoverable errors so the
      rest of the translation still proceeds
    """
    if not text or not text.strip():
        return text

    chunks = _split_into_sentence_chunks(text)
    translated_parts = []

    for chunk in chunks:
        for attempt in range(max_retries):
            try:
                result = GoogleTranslator(
                    source=source_lang, target='en'
                ).translate(chunk)
                translated_parts.append(result if result else chunk)
                # Polite delay between chunks
                time.sleep(0.4 + random.uniform(0, 0.2))
                break
            except Exception as e:
                err_str = str(e).lower()
                if 'rate' in err_str or '429' in err_str or 'quota' in err_str:
                    wait = (2 ** attempt) + random.uniform(0, 1)
                    print(f'    Rate limit hit — waiting {wait:.1f}s (attempt {attempt+1})')
                    time.sleep(wait)
                elif attempt == max_retries - 1:
                    # Exhausted retries — keep original to avoid data loss
                    print(f'    Translation failed after {max_retries} attempts: {str(e)[:80]}')
                    translated_parts.append(chunk)
                    break
                else:
                    time.sleep(1)

    return ' '.join(translated_parts)


# ── Main translation loop ─────────────────────────────────────
df_corpus = pd.read_csv(OUTPUT_FILE)

# Audit columns
if 'translated'        not in df_corpus.columns: df_corpus['translated']        = False
if 'detected_lang'     not in df_corpus.columns: df_corpus['detected_lang']     = ''
if 'translated_fields' not in df_corpus.columns: df_corpus['translated_fields'] = ''

translate_count  = 0
skip_english     = 0
already_done     = 0

print('Phase 1: Detecting languages and translating non-English fields...')
print(f'Fields targeted: {TRANSLATE_FIELDS}\n')

for idx, row in tqdm(df_corpus.iterrows(), total=len(df_corpus), desc='Translating'):

    # Skip rows already translated in a previous run
    if row.get('translated') is True:
        already_done += 1
        continue

    changed_fields = []

    # --- detect dominant language from the richest available field ---
    # Use description first (longest), then claims, then abstract.
    # Avoids false 'en' detection from short English patent headers.
    lang = 'en'
    for probe_field in ['description', 'claims', 'abstract', 'title']:
        probe_text = str(row.get(probe_field, '') or '').strip()
        if len(probe_text) >= 80:
            lang = detect_language(probe_text)
            break

    df_corpus.at[idx, 'detected_lang'] = lang

    if lang == 'en':
        skip_english += 1
        continue

    # --- translate each field independently -------------------------
    for field in TRANSLATE_FIELDS:
        value = str(row.get(field, '') or '').strip()
        if not value or len(value) < 15:
            continue

        # Re-check per field: abstract might be English even if
        # description is German (bilingual patent families are common)
        field_lang = detect_language(value) if len(value) >= 40 else lang
        if field_lang == 'en':
            continue

        translated = translate_to_english(value, source_lang=field_lang)
        if translated and translated.strip() != value:
            df_corpus.at[idx, field] = translated
            changed_fields.append(field)

    if changed_fields:
        # Rebuild full_text from the now-English fields
        df_corpus.at[idx, 'full_text'] = '\n\n'.join(
            str(df_corpus.at[idx, f]) for f in TEXT_FIELDS
            if str(df_corpus.at[idx, f]).strip()
        )
        df_corpus.at[idx, 'translated']        = True
        df_corpus.at[idx, 'translated_fields'] = ', '.join(changed_fields)
        translate_count += 1

    if translate_count > 0 and translate_count % 25 == 0:
        df_corpus.to_csv(OUTPUT_FILE, index=False)
        print(f'  Checkpoint: {translate_count} patents translated so far')

df_corpus.to_csv(OUTPUT_FILE, index=False)

print(f'\nTranslation complete.')
print(f'  Translated        : {translate_count}')
print(f'  Already English   : {skip_english}')
print(f'  Previously done   : {already_done}')

if translate_count > 0:
    print('\nTranslated patents by jurisdiction:')
    print(df_corpus[df_corpus['translated'] == True]
          .groupby('jurisdiction').size()
          .sort_values(ascending=False).to_string())
    print('\nLanguages detected across full corpus:')
    print(df_corpus['detected_lang'].value_counts().to_string())

# ── Phase 2: Re-extract processing snippets from translated text ─
# CRITICAL: this must run AFTER translation so that snippets are
# always drawn from English text and regex keywords can match.
print('\nPhase 2: Re-extracting processing snippets from translated text...')

df_corpus['processing_snippets'] = df_corpus.apply(
    lambda r: extract_processing_snippets(
        ' '.join(str(r.get(f, '')) for f in ['description', 'claims', 'abstract'])
    ),
    axis=1
)

has_snippets = df_corpus['processing_snippets'].str.len().gt(20)
df_corpus.to_csv(OUTPUT_FILE, index=False)
print(f'Processing snippets after translation: {has_snippets.sum()} / {len(df_corpus)}')

In [ ]:
# ──────────────────────────────────────────────────────────────
# BLOCK 7: Diagnostics & corpus summary
# ──────────────────────────────────────────────────────────────
import pandas as pd

df_corpus = pd.read_csv(OUTPUT_FILE)

has_title    = df_corpus['title'].notna()       & df_corpus['title'].str.len().gt(0)
has_abstract = df_corpus['abstract'].notna()    & df_corpus['abstract'].str.len().gt(20)
has_claims   = df_corpus['claims'].notna()      & df_corpus['claims'].str.len().gt(20)
has_desc     = df_corpus['description'].notna() & df_corpus['description'].str.len().gt(20)
has_proc     = df_corpus['processing_snippets'].notna() & df_corpus['processing_snippets'].str.len().gt(20)
has_any      = df_corpus['full_text'].notna()   & df_corpus['full_text'].str.len().gt(20)

print('=== Corpus Summary ===')
print(f'Total records            : {len(df_corpus)}')
print(f'Unique Lens IDs          : {df_corpus["lens_id"].nunique()}')
print(f'With title               : {has_title.sum()}')
print(f'With abstract            : {has_abstract.sum()}')
print(f'With claims              : {has_claims.sum()}')
print(f'With description         : {has_desc.sum()}')
print(f'With processing snippets : {has_proc.sum()}')
print(f'With any text            : {has_any.sum()}')
print(f'Translated               : {df_corpus["translated"].sum()}')

print('\n=== By jurisdiction ===')
print(df_corpus.groupby('jurisdiction').size().sort_values(ascending=False).head(10).to_string())

print('\n=== By role ===')
print(df_corpus.groupby('role').size().to_string())

print('\n── Sample with processing snippets ──')
sample = df_corpus[has_proc].iloc[0]
print(f'  Lens ID    : {sample["lens_id"]}')
print(f'  Title      : {str(sample["title"])[:80]}')
print(f'  IPC codes  : {str(sample["ipc_codes"])[:80]}')
print(f'  Abstract   : {str(sample["abstract"])[:200]}')
print(f'  Processing : {str(sample["processing_snippets"])[:400]}')

# ── Export MatScibert-ready slice ─────────────────────────────
MATSCIBERT_FILE = os.path.join(DRIVE_ROOT, 'opv_matscibert_input.csv')
matscibert_cols = ['lens_id', 'doc_key', 'jurisdiction', 'year', 'ipc_codes',
                   'title', 'abstract', 'claims', 'description', 'processing_snippets']
df_matscibert = df_corpus[has_proc][matscibert_cols].drop_duplicates(subset='lens_id')
df_matscibert.to_csv(MATSCIBERT_FILE, index=False)
print(f'\nMatScibert input saved to: {MATSCIBERT_FILE}')
print(f'Rows: {len(df_matscibert)}')

In [ ]:
# ──────────────────────────────────────────────────────────────
# BLOCK 8: MatScibert usage preview
#
# Shows how to pass the extracted processing_snippets into
# MatScibert for NER token extraction of OPV constraints.
# ──────────────────────────────────────────────────────────────

# Install MatScibert if not present
# !pip install -q transformers torch

MATSCIBERT_MODEL = 'm3rg-iitd/matscibert'

PREVIEW_CODE = '''
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch, pandas as pd

# Load MatScibert
tokenizer = AutoTokenizer.from_pretrained("m3rg-iitd/matscibert")
model = AutoModelForTokenClassification.from_pretrained("m3rg-iitd/matscibert")
model.eval()

df = pd.read_csv("opv_matscibert_input.csv")

results = []
for _, row in df.iterrows():
    text = str(row["processing_snippets"])
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
    labels = [model.config.id2label[p] for p in predictions]
    results.append({
        "lens_id": row["lens_id"],
        "tokens": list(zip(tokens, labels))
    })

# Filter to material/property/process entities
OPV_ENTITY_TYPES = {"MAT", "PRO", "DSC", "CMT", "FORM", "NUM", "UNIT"}
for r in results[:3]:
    print(r["lens_id"])
    print([(t, l) for t, l in r["tokens"]
           if any(e in l for e in OPV_ENTITY_TYPES)])
    print()
'''

print('MatScibert usage preview (not executed — install transformers to run):')
print(PREVIEW_CODE)